In [1]:
import argparse
import csv
import importlib
import re
import sys
from collections import defaultdict
from pathlib import Path
import pandas as pd
from dataclasses import dataclass
from IPython.display import display


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / 'tools' / 'fsdb_cli').exists():
            return path
    raise RuntimeError('Repository root not found')


REPO_ROOT = find_repo_root(Path.cwd())
LATENCY_DIR = REPO_ROOT / 'analysis_workspace' / 'latency'
TOOLS_DIR = REPO_ROOT / 'tools'
for path in (LATENCY_DIR, TOOLS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import fsdb_cli as fsdb
import cycle_util
cycle_util = importlib.reload(cycle_util)


In [2]:
@dataclass
class Workload:
    name: str = 'kernel'
    kernel_type: str = 'generic'
    m: int = 0
    n: int = 0
    k: int = 0

    @property
    def is_fpxint_mxu(self):
        return self.kernel_type == 'fpxint_mxu'

    @property
    def is_fpxfp_tcu(self):
        return self.kernel_type == 'fpxfp_tcu'

    @property
    def is_gemm(self):
        return self.is_fpxint_mxu or self.is_fpxfp_tcu


@dataclass
class RunResult:
    workload: Workload
    fsdb_path: Path
    simv_log: Path
    elf_path: Path
    phases: pd.DataFrame
    sync_wait: pd.DataFrame | None = None
    mpm_accel: pd.DataFrame | None = None
    util_summary: pd.DataFrame | None = None
    fire_intervals: pd.DataFrame | None = None
    tcu_summary: pd.DataFrame | None = None
    tcu_breakdown: pd.DataFrame | None = None
    run_metadata: pd.DataFrame | None = None


In [3]:
def _path_or_default(value, default):
    if value is None or value == '':
        return Path(default)
    return Path(value)


def _default_simv_log(fsdb_path):
    return Path(fsdb_path).with_name('simv.log')


def _default_trace_dir(fsdb_path):
    fsdb_path = Path(fsdb_path)
    if fsdb_path.parent.name == 'xrtsim_vcs':
        return fsdb_path.parent.parent
    return fsdb_path.parent


def _default_elf_path(workload, trace_dir=None):
    if trace_dir is not None:
        trace_elf = Path(trace_dir) / 'kernel.elf'
        if trace_elf.exists():
            return trace_elf
    if workload is not None and workload.is_fpxfp_tcu:
        return cycle_util.DEFAULT_TCU_KERNEL_ELF
    return cycle_util.DEFAULT_KERNEL_ELF


def _mpm_value(mpm_accel, section, metric, default=0.0):
    if mpm_accel is None or mpm_accel.empty:
        return default
    value = mpm_accel.loc[(mpm_accel['section'] == section) & (mpm_accel['metric'] == metric), 'value']
    return default if value.empty else value.iloc[0]


def _phase_value(phases, phase, default=0.0):
    if phases is None or phases.empty:
        return default
    value = phases.loc[phases['phase'] == phase, 'cycles']
    return default if value.empty else value.iloc[0]


def _safe_pct(num, den):
    return 0.0 if den == 0 else 100.0 * num / den


def build_util_summary(phases, mpm_accel):
    """Build a compact utilization table from fpint_naive MXU/dcache-DMA counters."""
    if mpm_accel is None or mpm_accel.empty:
        return pd.DataFrame()

    rows = []

    def add(section, metric, value, unit='pct'):
        rows.append({'section': section, 'metric': metric, 'value': value, 'unit': unit})

    kernel_busy = _phase_value(phases, 'kernel_busy')
    user_body = _phase_value(phases, 'user_kernel_body')
    gemm_total = _mpm_value(mpm_accel, 'mxu', 'gemm_total_cycles')
    gemm_compute = _mpm_value(mpm_accel, 'mxu', 'gemm_compute_cycles')
    gemm_stall = _mpm_value(mpm_accel, 'mxu', 'gemm_stall_cycles')
    dcache_dma_active = _mpm_value(mpm_accel, 'dcache_dma', 'active_cycles')

    add('mxu', 'active_pct_kernel_busy', _safe_pct(gemm_total, kernel_busy))
    add('mxu', 'active_pct_user_kernel_body', _safe_pct(gemm_total, user_body))
    add('mxu', 'compute_pct_gemm_total', _safe_pct(gemm_compute, gemm_total))
    add('mxu', 'stall_pct_gemm_total', _safe_pct(gemm_stall, gemm_total))
    add('mxu', 'flops_per_gemm_cycle', _mpm_value(mpm_accel, 'mxu', 'achieved_flops_per_cycle_total'), 'flop/cycle')
    add('mxu', 'overlap_dma_mxu_pct_total', _mpm_value(mpm_accel, 'mxu', 'overlap_dma_mxu_pct_total'))

    for port in ('input', 'weight', 'psum', 'output'):
        add('mxu_port', f'{port}_util_pct_total', _mpm_value(mpm_accel, 'mxu', f'{port}_util_pct_total'))
        add('mxu_port', f'{port}_util_pct_compute', _mpm_value(mpm_accel, 'mxu', f'{port}_util_pct_compute'))
        add('mxu_port', f'{port}_stall_pct_activity', _mpm_value(mpm_accel, 'mxu', f'{port}_stall_pct_activity'))

    add('dcache_dma', 'active_pct_gemm_total', _safe_pct(dcache_dma_active, gemm_total))
    add('dcache_dma', 'active_pct_user_kernel_body', _safe_pct(dcache_dma_active, user_body))
    add('dcache_dma', 'util_pct_busy', _mpm_value(mpm_accel, 'dcache_dma', 'util_pct_busy'))
    add('dcache_dma', 'util_pct_total', _mpm_value(mpm_accel, 'dcache_dma', 'util_pct_total'))
    add('dcache_dma', 'bandwidth_bytes_per_active_cycle', _mpm_value(mpm_accel, 'dcache_dma', 'bandwidth_bytes_per_active_cycle'), 'B/cycle')
    add('dcache_dma', 'bandwidth_bytes_per_busy_cycle', _mpm_value(mpm_accel, 'dcache_dma', 'bandwidth_bytes_per_busy_cycle'), 'B/cycle')

    return pd.DataFrame(rows)


def run(
    fsdb_path=None,
    workload=None,
    *,
    simv_log=None,
    elf_path=None,
    paths=None,
    include_mxu=None,
    include_tcu=None,
    show=True,
    strict=False,
    include_intervals=None,
    interval_groups=None,
    interval_specs=None,
):
    """Run reusable latency analysis for fpint_naive FPxINT/MXU and FPxFP/TCU kernels."""
    workload = workload or Workload()
    paths = paths or cycle_util.DEFAULT_GEMM_PATHS
    fsdb_path = _path_or_default(fsdb_path, paths.fsdb_path)
    simv_log = _path_or_default(simv_log, _default_simv_log(fsdb_path))
    elf_path = _path_or_default(elf_path, _default_elf_path(workload))
    include_mxu = workload.is_fpxint_mxu if include_mxu is None else include_mxu
    include_tcu = workload.is_fpxfp_tcu if include_tcu is None else include_tcu
    include_intervals = include_mxu if include_intervals is None else include_intervals

    phases = cycle_util.analyze_kernel_agnostic_phases(
        fsdb_path=fsdb_path,
        simv_log=simv_log,
        elf_path=elf_path,
        paths=paths,
        strict=strict,
    )

    sync_wait = None
    mpm_accel = None
    util_summary = None
    fire_intervals = None
    tcu_summary = None
    tcu_breakdown = None
    run_metadata = cycle_util.parse_run_metadata(
        trace_dir=_default_trace_dir(fsdb_path),
        simv_log=simv_log,
    )

    if include_mxu:
        sync_wait = cycle_util.analyze_sync_wait(fsdb_path, paths)
        mpm_accel = cycle_util.analyze_mpm_accel(fsdb_path, paths, strict=strict)
        util_summary = build_util_summary(phases, mpm_accel)

    if include_intervals:
        fire_intervals = cycle_util.analyze_signal_intervals(
            fsdb_path=fsdb_path,
            paths=paths,
            specs=interval_specs,
            groups=interval_groups,
            kind='fire',
            strict=strict,
        )

    if include_tcu:
        tcu_summary = cycle_util.analyze_tcu_trace(
            fsdb_path=fsdb_path,
            simv_log=simv_log,
            strict=strict,
        )
        tcu_system = cycle_util.analyze_tcu_system_metrics(
            fsdb_path=fsdb_path,
            phases=phases,
            paths=paths,
            strict=strict,
        )
        tcu_summary = pd.concat([tcu_summary, tcu_system], ignore_index=True)
        tcu_breakdown = cycle_util.analyze_tcu_kernel_breakdown(
            simv_log=simv_log,
            elf_path=elf_path,
            phases=phases,
            strict=strict,
        )

    result = RunResult(
        workload=workload,
        fsdb_path=fsdb_path,
        simv_log=simv_log,
        elf_path=elf_path,
        phases=phases,
        sync_wait=sync_wait,
        mpm_accel=mpm_accel,
        util_summary=util_summary,
        fire_intervals=fire_intervals,
        tcu_summary=tcu_summary,
        tcu_breakdown=tcu_breakdown,
        run_metadata=run_metadata,
    )

    if show:
        print(f'workload={workload.name} type={workload.kernel_type} m={workload.m} n={workload.n} k={workload.k}')
        print(f'fsdb={fsdb_path}')
        print(f'simv_log={simv_log}')
        print(f'elf={elf_path}')

        phase_cols = [
            'phase', 'start_cycle', 'end_cycle', 'cycles', 'count',
            'dispatch_count', 'commit_count', 'symbols', 'note',
        ]
        display(phases[[col for col in phase_cols if col in phases.columns]])

        for table in (run_metadata, sync_wait, mpm_accel, util_summary, fire_intervals, tcu_summary, tcu_breakdown):
            if table is not None:
                display(table)

    return result


In [4]:
def parse_naive_workload(trace_dir):
    trace_dir = Path(trace_dir)
    match = re.search(r'm(\d+)_k(\d+)_n(\d+)', trace_dir.name)
    m = k = n = 0
    if match:
        m, k, n = map(int, match.groups())

    name = trace_dir.name
    if name.startswith('sgemm_tcu'):
        kernel_type = 'fpxfp_tcu'
    elif name.startswith(('fpint_naive', 'fpint_improve')):
        kernel_type = 'fpxint_mxu'
    else:
        kernel_type = 'generic'
    return Workload(name=name, kernel_type=kernel_type, m=m, n=n, k=k)


def _annotate(df, workload):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if 'kind' in out.columns:
        out = out.rename(columns={'kind': 'signal_kind'})
    out.insert(0, 'trace', workload.name)
    out.insert(1, 'kind', workload.kernel_type)
    out.insert(2, 'm', workload.m)
    out.insert(3, 'k', workload.k)
    out.insert(4, 'n', workload.n)
    return out


def _annotate_metadata(df, workload):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if 'trace' not in out.columns:
        out.insert(0, 'trace', workload.name)
    else:
        out['trace'] = out['trace'].fillna(workload.name)
    insert_at = 1
    for col, value in (('kind', workload.kernel_type), ('m', workload.m), ('k', workload.k), ('n', workload.n)):
        if col not in out.columns:
            out.insert(insert_at, col, value)
            insert_at += 1
    return out


def _read_log_sample(path, sample_bytes=131072):
    path = Path(path)
    size = path.stat().st_size
    with path.open('rb') as f:
        head = f.read(sample_bytes)
        if size > sample_bytes:
            f.seek(max(0, size - sample_bytes))
            tail = f.read(sample_bytes)
        else:
            tail = b''
    return (head + b'\n' + tail).decode(errors='ignore')


def _detect_failed_trace(trace_dir, simv_log):
    checks = []
    run_log = Path(trace_dir) / 'run.log'
    if run_log.exists():
        checks.append(('run_log', _read_log_sample(run_log)))
    if Path(simv_log).exists():
        checks.append(('simv_log', _read_log_sample(simv_log)))

    for source, text in checks:
        if 'Address already in use' in text:
            return f'run_failed:{source}:address_in_use'
        if 'Fatal:' in text:
            return f'run_failed:{source}:fatal'

    if run_log.exists():
        text = _read_log_sample(run_log)
        passed = 'PASSED' in text or 'PASSED!' in text
        failed = 'Error:' in text or 'make: ***' in text or 'connect timeout' in text
        if failed and not passed:
            return 'run_failed:run_log:error'
    return None


def _csv_ready(df):
    if df is None:
        return df
    out = df.copy()
    for col in ('count', 'dispatch_count', 'commit_count'):
        if col in out.columns:
            out[col] = out[col].fillna(0)
    for col in out.select_dtypes(include='object').columns:
        out[col] = out[col].fillna('-').replace('', '-')
    return out


def _compact_phase_view(phase_summary):
    order = [
        'runtime_bootstrap_to_user', 'tag_init', 'warp_spawn', 'user_kernel_body',
        'runtime_exit', 'perf_dump', 'exit_final_to_fence', 'fence_wait',
        'cache_flush_tags', 'host_done_polling',
    ]
    table = phase_summary[phase_summary['phase'].isin(order)].pivot_table(
        index='trace', columns='phase', values='cycles', aggfunc='first'
    )
    return table[[col for col in order if col in table.columns]].fillna(0)


def _compact_sync_view(sync_summary):
    if sync_summary.empty:
        return sync_summary
    order = ['G0', 'G1', 'O', 'W0', 'SZ0', 'W1', 'SZ1', 'T0', 'T1']
    table = sync_summary.pivot_table(
        index='trace', columns='wait_reg_name', values='cycles', aggfunc='first'
    )
    return table[[col for col in order if col in table.columns]].fillna(0)


def _compact_mpm_view(mpm_summary):
    if mpm_summary.empty:
        return mpm_summary

    metrics = [
        ('mxu', 'gemm_total_cycles', 'gemm_total'),
        ('mxu', 'gemm_compute_cycles', 'compute'),
        ('mxu', 'gemm_stall_cycles', 'stall'),
        ('mxu', 'gemm_job_count', 'jobs'),
        ('mxu', 'mxu_mac_count', 'macs'),
        ('mxu', 'lmem_rd_bytes', 'lmem_rd_bytes'),
        ('mxu', 'lmem_wr_bytes', 'lmem_wr_bytes'),
        ('mxu', 'achieved_flops_per_cycle_total', 'flops_per_cycle'),
        ('dcache_dma', 'rd_bytes', 'global_rd_bytes'),
        ('dcache_dma', 'wr_bytes', 'global_wr_bytes'),
        ('dcache_dma', 'global_dma_bytes', 'global_dma_bytes'),
        ('dcache_dma', 'active_cycles', 'dcache_dma_active'),
        ('dcache_dma', 'wait_dcache', 'dcache_dma_wait_dcache'),
        ('dcache_dma', 'wait_lmem', 'dcache_dma_wait_lmem'),
        ('dcache_dma', 'bandwidth_bytes_per_active_cycle', 'dcache_dma_bw_active_Bpc'),
        ('roofline', 'operational_intensity_global_dma', 'oi_global_dma'),
    ]

    rows = []
    for trace, sub in mpm_summary.groupby('trace', sort=True):
        row = {'trace': trace}
        for section, metric, name in metrics:
            value = sub.loc[(sub['section'] == section) & (sub['metric'] == metric), 'value']
            row[name] = None if value.empty else value.iloc[0]
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace').fillna(0)


def _compact_util_view(util_summary):
    if util_summary.empty:
        return util_summary

    metrics = [
        ('mxu', 'active_pct_user_kernel_body', 'mxu_active_user_pct'),
        ('mxu', 'compute_pct_gemm_total', 'mxu_compute_pct'),
        ('mxu', 'stall_pct_gemm_total', 'mxu_stall_pct'),
        ('mxu', 'flops_per_gemm_cycle', 'flops_per_cycle'),
        ('mxu_port', 'input_util_pct_compute', 'mxu_input_util_compute'),
        ('mxu_port', 'weight_util_pct_compute', 'mxu_weight_util_compute'),
        ('mxu_port', 'psum_util_pct_compute', 'mxu_psum_util_compute'),
        ('mxu_port', 'output_util_pct_compute', 'mxu_output_util_compute'),
        ('dcache_dma', 'active_pct_user_kernel_body', 'dcache_dma_active_user_pct'),
        ('dcache_dma', 'bandwidth_bytes_per_active_cycle', 'dcache_dma_bw_active_Bpc'),
    ]

    rows = []
    for trace, sub in util_summary.groupby('trace', sort=True):
        row = {'trace': trace}
        for section, metric, name in metrics:
            value = sub.loc[(sub['section'] == section) & (sub['metric'] == metric), 'value']
            row[name] = None if value.empty else value.iloc[0]
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace').fillna(0)


def _compact_fire_interval_view(fire_intervals):
    if fire_intervals.empty:
        return fire_intervals

    preferred = [
        ('mxu', 'input'),
        ('mxu', 'weight'),
        ('mxu', 'psum'),
        ('mxu', 'output'),
        ('dcache_dma', 'src_rd_req'),
        ('dcache_dma', 'src_rd_data'),
        ('dcache_dma', 'dst_wr'),
        ('ldma_input', 'src_rd_req'),
        ('ldma_input', 'dst_wr'),
        ('ldma_weight', 'src_rd_req'),
        ('ldma_weight', 'dst_wr'),
        ('ldma_sz', 'src_rd_req'),
        ('ldma_sz', 'dst_wr'),
        ('ldma_output', 'src_rd_req'),
        ('ldma_output', 'dst_wr'),
    ]

    rows = []
    for trace, sub in fire_intervals.groupby('trace', sort=True):
        row = {'trace': trace}
        for section, stream in preferred:
            match = sub[(sub['section'] == section) & (sub['stream'] == stream)]
            prefix = f'{section}_{stream}'
            if match.empty:
                row[f'{prefix}_p50'] = None
                row[f'{prefix}_p90'] = None
                row[f'{prefix}_max_burst'] = None
                row[f'{prefix}_consec_pct'] = None
            else:
                item = match.iloc[0]
                row[f'{prefix}_p50'] = item['p50_interval']
                row[f'{prefix}_p90'] = item['p90_interval']
                row[f'{prefix}_max_burst'] = item['max_burst_len']
                row[f'{prefix}_consec_pct'] = item['consecutive_interval_pct']
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace').fillna(0)


def _compact_tcu_view(tcu_summary):
    if tcu_summary.empty:
        return tcu_summary
    metrics = [
        ('tcu', 'dispatch_count', 'dispatch_count'),
        ('tcu', 'commit_count', 'commit_count'),
        ('tcu', 'latency_mean', 'latency_mean'),
        ('tcu', 'latency_p50', 'latency_p50'),
        ('tcu', 'latency_p90', 'latency_p90'),
        ('tcu', 'dispatch_p50_interval', 'dispatch_p50_interval'),
        ('tcu', 'commit_p50_interval', 'commit_p50_interval'),
        ('tcu_pe', 'pe_lane_util_pct_busy', 'tcu_pe_util_busy_pct'),
        ('tcu_pe', 'pe_lane_util_pct_user_kernel_body', 'tcu_pe_util_user_pct'),
        ('tcu_pe', 'active_pct_busy', 'tcu_pe_active_busy_pct'),
        ('tcu_pe', 'active_pct_user_kernel_body', 'tcu_pe_active_user_pct'),
        ('tcu_fp', 'util_pct_busy', 'tcu_fp_util_busy_pct'),
        ('tcu_fp', 'util_pct_user_kernel_body', 'tcu_fp_util_user_pct'),
        ('tcu_fp', 'fire_count', 'tcu_fp_fire_count'),
        ('tcu_fp', 'fire_p50_interval', 'tcu_fp_fire_p50_int'),
        ('tcu_int', 'util_pct_busy', 'tcu_int_util_busy_pct'),
        ('tcu_int', 'util_pct_user_kernel_body', 'tcu_int_util_user_pct'),
        ('tcu_int', 'fire_count', 'tcu_int_fire_count'),
        ('tcu_int', 'fire_p50_interval', 'tcu_int_fire_p50_int'),
        ('hbm_axi', 'total_bytes', 'hbm_bytes'),
        ('hbm_axi', 'active_pct_busy', 'hbm_util_busy_pct'),
        ('hbm_axi', 'active_pct_user_kernel_body', 'hbm_util_user_pct'),
        ('hbm_axi', 'bandwidth_bytes_per_busy_cycle', 'hbm_bw_busy_Bpc'),
        ('hbm_axi', 'bandwidth_bytes_per_user_cycle', 'hbm_bw_user_Bpc'),
        ('hbm_axi', 'read_latency_mean', 'hbm_read_lat_mean'),
        ('hbm_axi', 'read_latency_p50', 'hbm_read_lat_p50'),
        ('hbm_axi', 'read_req_p50_interval', 'hbm_read_req_p50_int'),
        ('hbm_axi', 'read_rsp_p50_interval', 'hbm_read_rsp_p50_int'),
        ('hbm_axi', 'write_data_p50_interval', 'hbm_write_data_p50_int'),
        ('dcache', 'total_bytes', 'dcache_bytes'),
        ('dcache', 'active_pct_busy', 'dcache_util_busy_pct'),
        ('dcache', 'active_pct_user_kernel_body', 'dcache_util_user_pct'),
        ('dcache', 'bandwidth_bytes_per_busy_cycle', 'dcache_bw_busy_Bpc'),
        ('dcache', 'bandwidth_bytes_per_user_cycle', 'dcache_bw_user_Bpc'),
        ('dcache', 'read_latency_mean', 'dcache_read_lat_mean'),
        ('dcache', 'read_latency_p50', 'dcache_read_lat_p50'),
        ('dcache', 'read_req_p50_interval', 'dcache_read_req_p50_int'),
        ('dcache', 'rsp_p50_interval', 'dcache_rsp_p50_int'),
        ('dcache', 'write_req_p50_interval', 'dcache_write_req_p50_int'),
        ('lmem', 'total_bytes', 'lmem_bytes'),
        ('lmem', 'active_pct_busy', 'lmem_util_busy_pct'),
        ('lmem', 'active_pct_user_kernel_body', 'lmem_util_user_pct'),
        ('lmem', 'bandwidth_bytes_per_busy_cycle', 'lmem_bw_busy_Bpc'),
        ('lmem', 'bandwidth_bytes_per_user_cycle', 'lmem_bw_user_Bpc'),
        ('lmem', 'read_latency_mean', 'lmem_read_lat_mean'),
        ('lmem', 'read_latency_p50', 'lmem_read_lat_p50'),
        ('lmem', 'read_req_p50_interval', 'lmem_read_req_p50_int'),
        ('lmem', 'rsp_p50_interval', 'lmem_rsp_p50_int'),
        ('lmem', 'write_req_p50_interval', 'lmem_write_req_p50_int'),
    ]
    rows = []
    for trace, sub in tcu_summary.groupby('trace', sort=True):
        row = {'trace': trace}
        for section, metric, name in metrics:
            value = sub.loc[(sub['section'] == section) & (sub['metric'] == metric), 'value']
            row[name] = None if value.empty else value.iloc[0]
        rows.append(row)
    return pd.DataFrame(rows).set_index('trace').fillna(0)


def run_many_fpint_naive(
    log_root=None,
    patterns=('fpint_naive_*', 'fpint_improve_*', 'sgemm_tcu_*'),
    *,
    write_csv=True,
    output_dir=None,
    csv_prefix='fpint_naive',
    show=True,
    include_intervals=True,
    interval_groups=None,
    interval_specs=None,
    require_fsdb=True,
    strict=False,
):
    log_root = _path_or_default(log_root, REPO_ROOT / 'build' / 'logs')
    output_dir = _path_or_default(output_dir, LATENCY_DIR)
    csv_prefix = str(csv_prefix).strip() or 'fpint_naive'
    paths = cycle_util.DEFAULT_GEMM_PATHS

    if isinstance(patterns, str):
        patterns = (patterns,)
    traces = []
    for pattern in patterns:
        traces.extend(path for path in log_root.glob(pattern) if path.is_dir())
    traces = sorted(set(traces))
    if not traces:
        raise FileNotFoundError(f'No traces matched {patterns} under {log_root}')

    phase_rows = []
    sync_rows = []
    mpm_rows = []
    util_rows = []
    interval_rows = []
    tcu_rows = []
    tcu_breakdown_rows = []
    metadata_rows = []
    missing_rows = []
    results = {}

    for trace_dir in traces:
        workload = parse_naive_workload(trace_dir)
        if workload.kernel_type == 'generic':
            continue
        fsdb_path = trace_dir / 'xrtsim_vcs' / 'vcs_cosim.fsdb'
        simv_log = trace_dir / 'xrtsim_vcs' / 'simv.log'
        missing = []
        if not fsdb_path.exists():
            missing.append('fsdb')
        if not simv_log.exists():
            missing.append('simv_log')
        if missing:
            missing_rows.append({'trace': workload.name, 'missing': ','.join(missing), 'path': str(trace_dir)})
            if require_fsdb or 'simv_log' in missing:
                print(f'skip missing {missing}: {trace_dir}')
                continue
        failure = _detect_failed_trace(trace_dir, simv_log)
        if failure:
            missing_rows.append({'trace': workload.name, 'missing': failure, 'path': str(trace_dir)})
            print(f'skip failed {failure}: {trace_dir}')
            continue

        elf_path = _default_elf_path(workload, trace_dir)
        result = run(
            fsdb_path=fsdb_path,
            workload=workload,
            simv_log=simv_log,
            elf_path=elf_path,
            paths=paths,
            show=False,
            include_intervals=include_intervals and workload.is_fpxint_mxu,
            interval_groups=interval_groups,
            interval_specs=interval_specs,
            strict=strict,
        )
        results[workload.name] = result
        phase_rows.append(_annotate(result.phases, workload))
        sync_rows.append(_annotate(result.sync_wait, workload))
        mpm_rows.append(_annotate(result.mpm_accel, workload))
        util_rows.append(_annotate(result.util_summary, workload))
        interval_rows.append(_annotate(result.fire_intervals, workload))
        tcu_rows.append(_annotate(result.tcu_summary, workload))
        tcu_breakdown_rows.append(_annotate(result.tcu_breakdown, workload))
        metadata_rows.append(_annotate_metadata(result.run_metadata, workload))

    phase_summary = pd.concat(phase_rows, ignore_index=True) if phase_rows else pd.DataFrame()
    sync_summary = pd.concat(sync_rows, ignore_index=True) if sync_rows else pd.DataFrame()
    mpm_summary = pd.concat(mpm_rows, ignore_index=True) if mpm_rows else pd.DataFrame()
    util_summary = pd.concat(util_rows, ignore_index=True) if util_rows else pd.DataFrame()
    fire_interval_summary = pd.concat(interval_rows, ignore_index=True) if interval_rows else pd.DataFrame()
    tcu_summary = pd.concat(tcu_rows, ignore_index=True) if tcu_rows else pd.DataFrame()
    tcu_breakdown_summary = pd.concat(tcu_breakdown_rows, ignore_index=True) if tcu_breakdown_rows else pd.DataFrame()
    metadata_summary = pd.concat(metadata_rows, ignore_index=True) if metadata_rows else pd.DataFrame()
    missing_summary = pd.DataFrame(missing_rows, columns=['trace', 'missing', 'path'])

    phase_compact = _compact_phase_view(phase_summary) if not phase_summary.empty else pd.DataFrame()
    sync_compact = _compact_sync_view(sync_summary) if not sync_summary.empty else pd.DataFrame()
    mpm_compact = _compact_mpm_view(mpm_summary) if not mpm_summary.empty else pd.DataFrame()
    util_compact = _compact_util_view(util_summary) if not util_summary.empty else pd.DataFrame()
    fire_interval_compact = (
        _compact_fire_interval_view(fire_interval_summary)
        if not fire_interval_summary.empty else pd.DataFrame()
    )
    tcu_compact = cycle_util.build_tcu_compact_summary(
        phase_summary=phase_summary,
        tcu_summary=tcu_summary,
        tcu_breakdown=tcu_breakdown_summary,
        metadata_summary=metadata_summary,
    ) if not tcu_summary.empty else pd.DataFrame()

    if write_csv:
        output_dir.mkdir(parents=True, exist_ok=True)

        def write_table(frame, suffix, *, index=False, keep_empty=False):
            if frame is None:
                return
            if frame.empty and not keep_empty:
                return
            _csv_ready(frame).to_csv(output_dir / f'{csv_prefix}_{suffix}.csv', index=index)

        write_table(phase_summary, 'phase_summary')
        write_table(sync_summary, 'sync_wait_summary')
        write_table(mpm_summary, 'mpm_summary')
        write_table(util_summary, 'util_summary')
        write_table(fire_interval_summary, 'fire_interval_summary')
        write_table(tcu_summary, 'tcu_summary')
        write_table(tcu_breakdown_summary, 'tcu_breakdown_summary')
        write_table(metadata_summary, 'run_metadata')
        write_table(missing_summary, 'missing_traces', keep_empty=True)

        write_table(phase_compact, 'phase_compact', index=True)
        write_table(sync_compact, 'sync_wait_compact', index=True)
        write_table(mpm_compact, 'mpm_compact', index=True)
        write_table(util_compact, 'util_compact', index=True)
        write_table(fire_interval_compact, 'fire_interval_compact', index=True)
        write_table(tcu_compact, 'tcu_compact', index=True)

    if show:
        print(f'traces={len(results)} log_root={log_root}')
        for table in (phase_compact, sync_compact, mpm_compact, util_compact, fire_interval_compact, tcu_compact, metadata_summary, tcu_breakdown_summary, missing_summary):
            if table is not None and not table.empty:
                display(table)

    return {
        'results': results,
        'phase_summary': phase_summary,
        'sync_summary': sync_summary,
        'mpm_summary': mpm_summary,
        'util_summary': util_summary,
        'fire_interval_summary': fire_interval_summary,
        'tcu_summary': tcu_summary,
        'tcu_breakdown_summary': tcu_breakdown_summary,
        'metadata_summary': metadata_summary,
        'missing_summary': missing_summary,
        'phase_compact': phase_compact,
        'sync_compact': sync_compact,
        'mpm_compact': mpm_compact,
        'util_compact': util_compact,
        'fire_interval_compact': fire_interval_compact,
        'tcu_compact': tcu_compact,
    }


In [6]:
fpint_naive_results = run_many_fpint_naive(
    # patterns=('fpint_naive_*',),
    patterns=('fpint_naive_m1_k128_n64',),
    include_intervals=True,
    csv_prefix='fpint_naive',
    show=False,
)


FSDB: /home/jaeyongjang/project.local/vortex_naive/build/logs/fpint_naive_m1_k128_n64/xrtsim_vcs/vcs_cosim.fsdb
simv.log: /home/jaeyongjang/project.local/vortex_naive/build/logs/fpint_naive_m1_k128_n64/xrtsim_vcs/simv.log
ELF: /home/jaeyongjang/project.local/vortex_naive/build/logs/fpint_naive_m1_k128_n64/kernel.elf
Window: bt=None, et=None, clock_period_ps=10000
FSDB: /home/jaeyongjang/project.local/vortex_naive/build/logs/fpint_naive_m1_k128_n64/xrtsim_vcs/vcs_cosim.fsdb
Window: bt=None, et=None, time_unit=1ps
total_wait_active_cycles=7810
FSDB: /home/jaeyongjang/project.local/vortex_naive/build/logs/fpint_naive_m1_k128_n64/xrtsim_vcs/vcs_cosim.fsdb
Window: bt=None, et=None
FSDB: /home/jaeyongjang/project.local/vortex_naive/build/logs/fpint_naive_m1_k128_n64/xrtsim_vcs/vcs_cosim.fsdb
Window: bt=None, et=None, clock_period_ps=10000, kind=fire
burst_gap_cycles=1, sample_on_clk=False


In [7]:
fpint_phase_compact = fpint_naive_results['phase_compact']
fpint_mpm_compact = fpint_naive_results['mpm_compact']
fpint_util_compact = fpint_naive_results['util_compact']
fpint_fire_interval_compact = fpint_naive_results['fire_interval_compact']
fpint_sync_compact = fpint_naive_results['sync_compact']

display(fpint_phase_compact)

mpm_cols = [
    'gemm_total', 'compute', 'stall', 'jobs', 'macs', 'lmem_rd_bytes', 'lmem_wr_bytes',
    'flops_per_cycle', 'global_rd_bytes', 'global_wr_bytes', 'global_dma_bytes',
    'dcache_dma_active', 'dcache_dma_wait_dcache', 'dcache_dma_wait_lmem',
    'dcache_dma_bw_active_Bpc', 'oi_global_dma',
]
display(fpint_mpm_compact[[col for col in mpm_cols if col in fpint_mpm_compact.columns]])

util_cols = [
    'mxu_active_user_pct', 'mxu_compute_pct', 'mxu_stall_pct', 'flops_per_cycle',
    'mxu_input_util_compute', 'mxu_weight_util_compute',
    'mxu_psum_util_compute', 'mxu_output_util_compute',
    'dcache_dma_active_user_pct', 'dcache_dma_bw_active_Bpc',
]
display(fpint_util_compact[[col for col in util_cols if col in fpint_util_compact.columns]])

display(fpint_fire_interval_compact)
display(fpint_sync_compact)


phase,runtime_bootstrap_to_user,tag_init,warp_spawn,user_kernel_body,runtime_exit,perf_dump,exit_final_to_fence,fence_wait,cache_flush_tags,host_done_polling
trace,,,,,,,,,,
fpint_naive_m1_k128_n64,4525,2048,264,11356,488,562,605,2081,2048,5


,gemm_total,compute,stall,jobs,macs,lmem_rd_bytes,lmem_wr_bytes,flops_per_cycle,global_rd_bytes,global_wr_bytes,global_dma_bytes,dcache_dma_active,dcache_dma_wait_dcache,dcache_dma_wait_lmem,dcache_dma_bw_active_Bpc,oi_global_dma
trace,,,,,,,,,,,,,,,,
fpint_naive_m1_k128_n64,7906.0,510.0,8.0,8.0,2048.0,5632.0,128.0,0.518088,9472.0,128.0,9600.0,3272.0,0.0,0.0,2.933985,0.426667


,mxu_active_user_pct,mxu_compute_pct,mxu_stall_pct,flops_per_cycle,mxu_input_util_compute,mxu_weight_util_compute,mxu_psum_util_compute,mxu_output_util_compute,dcache_dma_active_user_pct,dcache_dma_bw_active_Bpc
trace,,,,,,,,,,
fpint_naive_m1_k128_n64,69.619584,6.450797,0.101189,0.518088,1.568627,50.196078,1.176471,0.392157,28.812962,2.933985


,mxu_input_p50,mxu_input_p90,mxu_input_max_burst,mxu_input_consec_pct,mxu_weight_p50,mxu_weight_p90,mxu_weight_max_burst,mxu_weight_consec_pct,mxu_psum_p50,mxu_psum_p90,...,ldma_sz_dst_wr_max_burst,ldma_sz_dst_wr_consec_pct,ldma_output_src_rd_req_p50,ldma_output_src_rd_req_p90,ldma_output_src_rd_req_max_burst,ldma_output_src_rd_req_consec_pct,ldma_output_dst_wr_p50,ldma_output_dst_wr_p90,ldma_output_dst_wr_max_burst,ldma_output_dst_wr_consec_pct
trace,,,,,,,,,,,,,,,,,,,,,
fpint_naive_m1_k128_n64,556.0,556.0,1,0.0,17.0,17.0,1,0.0,556.0,556.0,...,1,0.0,7.0,7.0,1,0.0,7.0,7.0,1,0.0


wait_reg_name,G0,G1,O,W0,W1,T0
trace,,,,,,
fpint_naive_m1_k128_n64,267,267,69,1987,1917,3303


In [8]:
tcu_nostall_results = run_many_fpint_naive(
    patterns=('sgemm_tcu_m64_k64_n64', 'sgemm_tcu_bcol_m64_k64_n64'),
    include_intervals=False,
    csv_prefix='tcu_nostall',
    show=False,
)


FSDB: /home/jaeyongjang/project.local/vortex_naive/build/logs/sgemm_tcu_bcol_m64_k64_n64/xrtsim_vcs/vcs_cosim.fsdb
simv.log: /home/jaeyongjang/project.local/vortex_naive/build/logs/sgemm_tcu_bcol_m64_k64_n64/xrtsim_vcs/simv.log
ELF: /home/jaeyongjang/project.local/vortex_naive/build/logs/sgemm_tcu_bcol_m64_k64_n64/kernel.elf
Window: bt=None, et=None, clock_period_ps=10000
FSDB: /home/jaeyongjang/project.local/vortex_naive/build/logs/sgemm_tcu_bcol_m64_k64_n64/xrtsim_vcs/vcs_cosim.fsdb
simv.log: /home/jaeyongjang/project.local/vortex_naive/build/logs/sgemm_tcu_bcol_m64_k64_n64/xrtsim_vcs/simv.log
FSDB: /home/jaeyongjang/project.local/vortex_naive/build/logs/sgemm_tcu_bcol_m64_k64_n64/xrtsim_vcs/vcs_cosim.fsdb
TCU system metric window: bt=67185000ps, et=924465000ps, busy_cycles=95482, user_cycles=85728
simv.log: /home/jaeyongjang/project.local/vortex_naive/build/logs/sgemm_tcu_bcol_m64_k64_n64/xrtsim_vcs/simv.log
ELF: /home/jaeyongjang/project.local/vortex_naive/build/logs/sgemm_tcu_bcol

In [8]:
tcu_compact = tcu_nostall_results['tcu_compact']
selected_cols = [
    'elapsed_time_ms', 'b_device_layout', 'dram_no_markov_stall',
    'user_kernel_body', 'latency_p50', 'latency_p90',
    'dispatch_p50_interval', 'dispatch_p99_interval', 'dispatch_max_interval',
    'tcu_fp_util_pct_user_kernel_body', 'tcu_int_util_pct_user_kernel_body',
    'hbm_read_latency_mean', 'hbm_read_latency_p90', 'hbm_read_latency_p99', 'hbm_read_latency_max',
    'dcache_read_latency_mean', 'dcache_read_latency_p90',
    'l2_read_count', 'l2_read_miss_count', 'l2_read_hit_rate_pct',
    'l2_write_count', 'l2_write_miss_count', 'l2_write_hit_rate_pct',
    'l2_overall_hit_rate_pct', 'l2_mshr_stalls_delta', 'l2_mem_stalls_delta',
    'l1_icache_read_hit_rate_pct', 'l1_dcache_hit_rate_available',
    'load_sync_pct_user_kernel_body', 'compute_pct_user_kernel_body', 'store_sync_pct_user_kernel_body',
    'tcu_gap_large_gap_count', 'tcu_gap_large_gap_cycles',
]
display(tcu_compact[[col for col in selected_cols if col in tcu_compact.columns]])


,elapsed_time_ms,b_device_layout,dram_no_markov_stall,user_kernel_body,latency_p50,latency_p90,dispatch_p50_interval,dispatch_p99_interval,dispatch_max_interval,tcu_fp_util_pct_user_kernel_body,...,l2_overall_hit_rate_pct,l2_mshr_stalls_delta,l2_mem_stalls_delta,l1_icache_read_hit_rate_pct,l1_dcache_hit_rate_available,load_sync_pct_user_kernel_body,compute_pct_user_kernel_body,store_sync_pct_user_kernel_body,tcu_gap_large_gap_count,tcu_gap_large_gap_cycles
trace,,,,,,,,,,,,,,,,,,,,,
sgemm_tcu_bcol_m64_k64_n64,110816,col-major,True,85728,9.0,11.0,2.0,20.0,2233.0,7.207680,...,87.293292,0.0,0.0,99.645500,0.0,71.830674,16.350551,11.520157,63.0,62621.0
sgemm_tcu_m64_k64_n64,149062,row-major,True,103771,10.0,14.0,2.0,358.2,3065.0,6.672384,...,90.680863,0.0,0.0,99.756707,0.0,73.293117,13.927783,11.498396,150.0,70349.0


In [9]:
breakdown_cols = [
    'trace', 'phase', 'dispatch_count', 'attributed_cycles', 'stall_or_gap_cycles',
    'pct_user_kernel_body', 'p50_issue_interval', 'p90_issue_interval',
    'p99_issue_interval', 'max_issue_interval', 'large_gap_count', 'large_gap_cycles', 'op_top',
]
display(tcu_nostall_results['tcu_breakdown_summary'][[
    col for col in breakdown_cols if col in tcu_nostall_results['tcu_breakdown_summary'].columns
]])


,trace,phase,dispatch_count,attributed_cycles,stall_or_gap_cycles,pct_user_kernel_body,p50_issue_interval,p90_issue_interval,p99_issue_interval,max_issue_interval,large_gap_count,large_gap_cycles,op_top
0,sgemm_tcu_bcol_m64_k64_n64,load_sync,12288,61579,49291.0,71.830674,2.0,13.0,19.0,2651,NaN,NaN,"FLW:4096, ADD:2240, SLLI:1664, MUL:640, FSGNJ...."
1,sgemm_tcu_bcol_m64_k64_n64,compute,8192,14017,5825.0,16.350551,2.0,2.0,2.0,2,NaN,NaN,"WMMA.fp16.fp32.0.0:1024, WMMA.fp16.fp32.0.1:10..."
2,sgemm_tcu_bcol_m64_k64_n64,compute_loop_overhead,256,256,0.0,0.298619,1.0,1.0,1.0,1,NaN,NaN,BLTU:256
3,sgemm_tcu_bcol_m64_k64_n64,store_sync,1984,9876,7892.0,11.520157,1.0,8.0,27.0,954,NaN,NaN,"FSW:512, SLLI:320, ADD:320, FLD:256, MUL:128, ..."
4,sgemm_tcu_bcol_m64_k64_n64,tcu_gap_summary,8191,79973,NaN,93.286907,2.0,2.0,20.0,2233,63.0,62621.0,">100_count:63, >100_cycles:62621"
5,sgemm_tcu_m64_k64_n64,load_sync,25664,76057,50393.0,73.293117,1.0,7.0,19.0,2679,NaN,NaN,"ADD:4800, SLLI:4672, FMV.S.X:2112, FLW:2048, L..."
6,sgemm_tcu_m64_k64_n64,compute,8192,14453,6261.0,13.927783,2.0,2.0,4.0,14,NaN,NaN,"WMMA.fp16.fp32.0.0:1024, WMMA.fp16.fp32.0.1:10..."
7,sgemm_tcu_m64_k64_n64,compute_loop_overhead,768,1329,561.0,1.280705,1.0,3.0,9.0,9,NaN,NaN,"ADDIW:256, ADD:256, BLTU:256"
8,sgemm_tcu_m64_k64_n64,store_sync,2432,11932,9500.0,11.498396,2.0,15.0,27.0,302,NaN,NaN,"FSW:512, LD:448, SLLI:320, ADD:320, FLD:256, M..."
9,sgemm_tcu_m64_k64_n64,tcu_gap_summary,8191,98039,NaN,94.476299,2.0,5.0,358.2,3065,150.0,70349.0,">100_count:150, >100_cycles:70349"
